# 생성모델 3종(GAN · VAE · Flow)으로 배우는 이상탐지 실습

이 노트북은 **정상 데이터만 보고 학습**한 뒤 이상치를 찾아내는 이상탐지(Anomaly Detection)의
핵심 아이디어를, 대표적인 생성모델 세 가지로 직접 비교해 보는 실습입니다.

| 모델 | 이상 점수(anomaly score)의 근거 | 성격 |
|------|-------------------------------|------|
| **VAE** | 재구성 오차 + KL (음의 ELBO) | 근사 우도 |
| **GAN** | 판별자(Discriminator)의 진짜/가짜 점수 | 밀도추정 아님(우회적) |
| **Flow (RealNVP)** | 정확한 음의 로그우도(NLL) | 정확한 밀도추정 |

**실습 시나리오**: 2차원 평면의 "두 개의 초승달(two moons)"을 **정상 분포**로 보고,
평면 전체에 균일하게 뿌려진 점들을 **이상치**로 둡니다.
2D이므로 모든 모델이 수 초 안에 학습되고, **이상 점수 지형(heatmap)을 눈으로 직접 비교**할 수 있습니다.

> Colab에서 `런타임 > 모두 실행`으로 바로 돌아갑니다. GPU 없이 CPU로도 1~2분이면 끝납니다.

---
**학습 목표**
1. "정상만 학습 → 점수화 → 임계값 판정"이라는 이상탐지의 일반 골격을 이해한다.
2. 세 모델이 **이상 점수를 정의하는 방식의 철학적 차이**를 체득한다.
3. 동일 데이터·동일 평가(ROC-AUC)로 세 접근을 정량/정성 비교한다.


## 1. 라이브러리 임포트 및 환경 설정

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.metrics import roc_auc_score, roc_curve

# 재현성을 위한 시드 고정
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)

# 참고: matplotlib 한글 폰트 문제를 피하기 위해 그래프 텍스트는 영어로 둡니다.
# (Colab에서 한글 라벨이 필요하면 NanumGothic 설치 후 plt.rc('font', family='NanumGothic') 사용)


## 2. 데이터 생성

이상탐지의 핵심 규칙: **학습에는 정상 데이터만 사용**합니다(이상치는 보지 않음).
테스트셋에만 정상 + 이상을 섞어 두고, 모델이 이상치를 골라내는지 평가합니다.


In [ ]:
# ===== 여기 수정 =====
N_NORMAL       = 1000   # 학습용 정상 데이터 개수
NOISE          = 0.08   # 두 초승달의 노이즈(작을수록 분포가 또렷함)
N_TEST_NORMAL  = 500    # 테스트용 정상
N_TEST_ANOMALY = 500    # 테스트용 이상
# =====================

def make_normal(n, noise=NOISE):
    X, _ = make_moons(n_samples=n, noise=noise)
    return X.astype(np.float32)

def make_anomaly(n, low=-2.0, high=3.0):
    # 평면 전체에 균일하게 뿌려진 점 = 이상치
    return np.random.uniform(low, high, size=(n, 2)).astype(np.float32)

# 학습: 정상만
X_train = make_normal(N_NORMAL)

# 테스트: 정상 + 이상 (라벨 0=정상, 1=이상)
X_test = np.concatenate([make_normal(N_TEST_NORMAL), make_anomaly(N_TEST_ANOMALY)], axis=0)
y_test = np.concatenate([np.zeros(N_TEST_NORMAL), np.ones(N_TEST_ANOMALY)])

# 표준화(학습 데이터 통계 기준) — 모든 모델 공통 전처리
mean = X_train.mean(0, keepdims=True)
std  = X_train.std(0, keepdims=True)
X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std

X_train_t = torch.tensor(X_train, device=device)
X_test_t  = torch.tensor(X_test,  device=device)

# 시각화
plt.figure(figsize=(6, 6))
plt.scatter(X_test[y_test==0,0], X_test[y_test==0,1], s=8, alpha=0.6, label="normal")
plt.scatter(X_test[y_test==1,0], X_test[y_test==1,1], s=8, alpha=0.3, c="red", label="anomaly")
plt.title("Test set: normal (moons) vs anomaly (uniform)")
plt.axis("equal"); plt.legend(); plt.show()


## 3. 공통 평가·시각화 함수

모든 모델은 `score_fn(x) -> 이상점수` 형태의 함수 하나로 통일합니다.
**규칙: 점수가 높을수록 "이상"**. 이렇게 통일해 두면 평가 코드를 그대로 재사용할 수 있습니다.


In [ ]:
@torch.no_grad()
def score_grid(score_fn, rng=(-3, 3), res=200):
    xs = np.linspace(*rng, res); ys = np.linspace(*rng, res)
    gx, gy = np.meshgrid(xs, ys)
    grid = np.stack([gx.ravel(), gy.ravel()], 1).astype(np.float32)
    s = score_fn(torch.tensor(grid, device=device)).cpu().numpy().reshape(res, res)
    return gx, gy, s

def plot_heatmap(score_fn, title):
    gx, gy, s = score_grid(score_fn)
    plt.figure(figsize=(6, 6))
    plt.contourf(gx, gy, s, levels=40, cmap="viridis")
    plt.colorbar(label="anomaly score (high = abnormal)")
    plt.scatter(X_train[:,0], X_train[:,1], s=4, c="white", alpha=0.4, label="train (normal)")
    plt.title(title); plt.axis("equal"); plt.legend(); plt.show()

def report_auc(score_fn, name):
    with torch.no_grad():
        scores = score_fn(X_test_t).cpu().numpy()
    auc = roc_auc_score(y_test, scores)
    print(f"[{name}] ROC-AUC = {auc:.4f}")
    return scores, auc


## 4. VAE — 재구성 오차 기반 이상탐지

**아이디어**: 정상 데이터로만 학습한 VAE는 정상은 잘 복원하지만, 처음 보는 이상치는
복원하지 못해 **재구성 오차가 커집니다**. 여기서는 재구성 오차에 KL 항을 더한
**음의 ELBO**를 이상 점수로 사용합니다(복원 실패 + 잠재공간에서 벗어남을 모두 반영).

$$\text{score}(x) = \underbrace{\|x-\hat{x}\|^2}_{\text{재구성 오차}} + \underbrace{D_{KL}(q(z|x)\,\|\,p(z))}_{\text{잠재공간 이탈}}$$


In [ ]:
class VAE(nn.Module):
    def __init__(self, in_dim=2, hidden=64, latent=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU())
        self.fc_mu     = nn.Linear(hidden, latent)
        self.fc_logvar = nn.Linear(hidden, latent)
        self.dec = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, in_dim))
    def encode(self, x):
        h = self.enc(x); return self.fc_mu(h), self.fc_logvar(h)
    def reparam(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.dec(self.reparam(mu, logvar)), mu, logvar

# ===== 여기 수정 =====
VAE_EPOCHS = 300
VAE_LR     = 1e-3
BETA       = 1.0    # KL 가중치(beta-VAE). 키우면 정규화 강해짐
# =====================

vae = VAE().to(device)
opt = optim.Adam(vae.parameters(), lr=VAE_LR)
for ep in range(VAE_EPOCHS):
    vae.train()
    recon, mu, logvar = vae(X_train_t)
    recon_loss = ((recon - X_train_t) ** 2).sum(1).mean()
    kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(1)).mean()
    loss = recon_loss + BETA * kl
    opt.zero_grad(); loss.backward(); opt.step()
    if (ep + 1) % 50 == 0:
        print(f"epoch {ep+1:3d}/{VAE_EPOCHS} | loss={loss.item():.3f} recon={recon_loss.item():.3f} kl={kl.item():.3f}")


In [ ]:
@torch.no_grad()
def vae_score(x):
    vae.eval()
    mu, logvar = vae.encode(x)
    recon = vae.dec(vae.reparam(mu, logvar))
    recon_err = ((recon - x) ** 2).sum(1)
    kl = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(1)
    return recon_err + kl                      # 음의 ELBO: 높을수록 이상

plot_heatmap(vae_score, "VAE: negative ELBO (recon error + KL)")
vae_scores, vae_auc = report_auc(vae_score, "VAE")


## 5. GAN — 판별자(Discriminator) 기반 이상탐지

**아이디어**: 정상 데이터로 GAN을 학습하면, 판별자는 "정상다움"을 어느 정도 학습합니다.
정상처럼 보이면 *진짜(real)*, 아니면 *가짜(fake)*로 분류하므로, **판별자 점수를 뒤집어**
이상 점수로 쓸 수 있습니다.

> **솔직한 한계(중요)**: GAN은 본래 **밀도를 추정하지 않습니다**. 판별자 점수는 학습이
> 끝나면 정상/가짜를 구분하는 *결정경계* 근처 정보만 담고 있어, 데이터에서 멀리 떨어진
> 이상치에 대한 점수가 불안정합니다. 또 mode collapse·판별자 과적합에 민감합니다.
> 아래 결과에서 VAE·Flow보다 점수 지형이 거칠고 AUC가 낮게 나오는 이유를 직접 관찰하세요.
> (실무에서는 AnoGAN, f-AnoGAN처럼 잠재공간 역추론을 추가해 보완합니다.)


In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=2, hidden=64, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, out_dim))
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, in_dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.LeakyReLU(0.2),
                                 nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
                                 nn.Linear(hidden, 1))      # logit 출력
    def forward(self, x): return self.net(x)

# ===== 여기 수정 =====
GAN_ITERS = 3000
GAN_LR    = 2e-4
Z_DIM     = 2
BATCH     = 256
# =====================

G = Generator(Z_DIM).to(device)
D = Discriminator().to(device)
opt_g = optim.Adam(G.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
opt_d = optim.Adam(D.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
n = X_train_t.shape[0]

for it in range(GAN_ITERS):
    # --- 판별자 D 학습 ---
    real = X_train_t[torch.randint(0, n, (BATCH,), device=device)]
    fake = G(torch.randn(BATCH, Z_DIM, device=device)).detach()
    d_real, d_fake = D(real), D(fake)
    loss_d = bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake))
    opt_d.zero_grad(); loss_d.backward(); opt_d.step()
    # --- 생성자 G 학습 ---
    gen = G(torch.randn(BATCH, Z_DIM, device=device))
    d_gen = D(gen)
    loss_g = bce(d_gen, torch.ones_like(d_gen))
    opt_g.zero_grad(); loss_g.backward(); opt_g.step()
    if (it + 1) % 500 == 0:
        print(f"iter {it+1:4d}/{GAN_ITERS} | loss_D={loss_d.item():.3f} loss_G={loss_g.item():.3f}")


In [ ]:
@torch.no_grad()
def gan_score(x):
    D.eval()
    logit = D(x).squeeze(1)
    return -logit          # 판별자가 '가짜'라고 볼수록(=logit 작을수록) 이상 점수 높음

plot_heatmap(gan_score, "GAN: discriminator-based score (least principled)")
gan_scores, gan_auc = report_auc(gan_score, "GAN")


## 6. Normalizing Flow (RealNVP) — 정확한 우도 기반 이상탐지

**아이디어**: Flow는 가역(invertible) 변환 $f$로 데이터 $x$를 단순한 기저분포(표준정규) $z$로
보냅니다. 변수변환 공식 덕분에 **정확한 로그우도**를 계산할 수 있습니다.

$$\log p(x) = \log p_z(f(x)) + \log\left|\det \frac{\partial f}{\partial x}\right|$$

정상 분포의 우도가 낮은 점일수록 이상치이므로, **음의 로그우도(NLL)** 를 이상 점수로 씁니다.
여기서는 affine coupling layer를 쌓은 **RealNVP**를 (라이브러리 없이) 직접 구현합니다.


In [ ]:
class AffineCoupling(nn.Module):
    # mask=1인 차원은 그대로 두고, mask=0인 차원만 아핀 변환한다.
    def __init__(self, dim=2, hidden=64, mask=None):
        super().__init__()
        self.register_buffer("mask", mask)
        def mlp():
            return nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, dim))
        self.scale_net = mlp()   # log-scale 출력
        self.trans_net = mlp()   # translation 출력

    def forward(self, x):
        x_m = x * self.mask
        s = torch.tanh(self.scale_net(x_m)) * (1 - self.mask)   # tanh로 수치 안정화
        t = self.trans_net(x_m) * (1 - self.mask)
        z = x_m + (1 - self.mask) * (x * torch.exp(s) + t)      # 정방향 x -> z
        log_det = s.sum(1)                                      # log|det J|
        return z, log_det

class RealNVP(nn.Module):
    def __init__(self, dim=2, hidden=64, n_coupling=6):
        super().__init__()
        # 마스크를 번갈아 적용해 두 차원이 서로를 변환하도록 함
        self.couplings = nn.ModuleList([
            AffineCoupling(dim, hidden,
                torch.tensor([i % 2, (i + 1) % 2], dtype=torch.float32))
            for i in range(n_coupling)])

    def log_prob(self, x):
        z, log_det = x, torch.zeros(x.shape[0], device=x.device)
        for c in self.couplings:
            z, ld = c(z); log_det = log_det + ld
        log_pz = -0.5 * (z ** 2 + np.log(2 * np.pi)).sum(1)     # 표준정규 기저분포
        return log_pz + log_det

# ===== 여기 수정 =====
FLOW_EPOCHS = 800
FLOW_LR     = 1e-3
N_COUPLING  = 6     # coupling layer 개수(많을수록 표현력↑, 과적합 주의)
# =====================

flow = RealNVP(n_coupling=N_COUPLING).to(device)
opt = optim.Adam(flow.parameters(), lr=FLOW_LR)
for ep in range(FLOW_EPOCHS):
    flow.train()
    loss = -flow.log_prob(X_train_t).mean()    # 음의 로그우도 최소화 = MLE
    opt.zero_grad(); loss.backward(); opt.step()
    if (ep + 1) % 100 == 0:
        print(f"epoch {ep+1:3d}/{FLOW_EPOCHS} | NLL={loss.item():.4f}")


In [ ]:
@torch.no_grad()
def flow_score(x):
    flow.eval()
    return -flow.log_prob(x)        # NLL: 높을수록 이상

plot_heatmap(flow_score, "Normalizing Flow: negative log-likelihood (exact density)")
flow_scores, flow_auc = report_auc(flow_score, "Flow")


## 7. 세 모델 비교

같은 데이터·같은 평가지표(ROC-AUC)로 세 접근을 정량/정성 비교합니다.
점수 지형(heatmap)이 **정상 분포의 모양을 얼마나 정확히 감싸는지**를 함께 보세요.


In [ ]:
print("===== ROC-AUC =====")
for name, auc in [("VAE", vae_auc), ("GAN", gan_auc), ("Flow", flow_auc)]:
    print(f"  {name:5s}: {auc:.4f}")

# ROC 곡선
plt.figure(figsize=(6, 6))
for name, sc in [("VAE", vae_scores), ("GAN", gan_scores), ("Flow", flow_scores)]:
    fpr, tpr, _ = roc_curve(y_test, sc)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, sc):.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC curves"); plt.legend(); plt.show()

# 점수 지형 나란히 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (fn, title) in zip(axes, [(vae_score,"VAE"), (gan_score,"GAN"), (flow_score,"Flow")]):
    gx, gy, s = score_grid(fn)
    cs = ax.contourf(gx, gy, s, levels=40, cmap="viridis")
    ax.scatter(X_train[:,0], X_train[:,1], s=3, c="white", alpha=0.4)
    ax.set_title(title); ax.axis("equal")
    fig.colorbar(cs, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("Anomaly score landscapes (white = normal training data)")
plt.show()


## 8. 토론 및 확장 과제

**관찰 포인트**
- **Flow**는 정상 분포(두 초승달)를 가장 정확히 감싸는 점수 지형을 보이며 보통 AUC가 가장 높습니다 → 정확한 밀도추정의 힘.
- **VAE**도 분포를 잘 감싸지만, 잠재공간 병목/근사 추론 때문에 경계가 다소 뭉툭합니다.
- **GAN**의 점수 지형은 거칠고 데이터에서 먼 영역의 점수가 불안정합니다 → 밀도추정기가 아니기 때문.

**직접 실험해 볼 것 (위 셀의 `여기 수정` 값을 바꿔보세요)**
1. `NOISE`를 0.2로 키우면 세 모델의 AUC가 어떻게 변하나? 분포가 흐려질 때 누가 더 강건한가?
2. Flow의 `N_COUPLING`을 2 vs 12로 바꾸면 점수 지형과 과적합 양상이 어떻게 달라지나?
3. VAE의 `BETA`를 0.1 / 5.0으로 바꾸면 재구성-정규화 균형이 점수에 주는 영향은?
4. 이상치 분포(`make_anomaly`)를 정상 근처로 좁히면(`low=-0.5, high=1.5`) 셋 다 난이도가 급상승합니다. 누가 가장 잘 버티나?

**현업·연구로의 연결**
- 산업 결함탐지(MVTec 등)에서는 동일한 골격을 고차원 이미지/특징에 적용합니다. 거기서도 **Flow 기반 밀도추정(예: normalizing-flow on pretrained features)** 이 강력한 이유가 여기서 보입니다.
- 단, 고차원에서는 Flow의 우도가 OOD 검출에 항상 신뢰되는 것은 아니라는 알려진 함정(typicality 문제)이 있으니, 단순 NLL 외에 특징공간·조건부 설계가 필요해집니다.
